# Predictive Maintenance Baseline Study

**Tools:** XGBoost, SHAP, scikit-learn


## imitations

1. FD001 is simulated and does not represent the full range of industrial operating regimes or failure modes.

2. A capped training target treats early-life RUL as equally healthy; this reduces sensitivity to arbitrary early-life labels but limits extrapolation above the cap.

3. Test labels are available only for the final observed cycle. Earlier test-cycle RUL values are reconstructed from the terminal label and cycle distance.

4. Confidence intervals quantify sampling variation across this fleet of engines; they are not individual-engine predictive intervals.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile

CONFIG = {
    "project_title": "Predictive Maintenance Baseline Study",
    "base_dir": Path.cwd(),
    "data_dir": Path.cwd() / "predictive_maintenance_data",
    "output_dir": Path.cwd() / "predictive_maintenance_outputs",
    "cache_zip": Path.cwd() / "predictive_maintenance_data" / "CMAPSSData.zip",
    "manual_zip": Path.cwd() / "CMAPSSData.zip",
    "manual_folder": Path.cwd() / "manual_fd001",
    "data_mode": "auto",  # auto, official, manual_zip, or manual_folder
    "nasa_resource_page": "https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data",
    "nasa_zip_url": "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "expected_files": ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"],
    "seed": 42,
    "validation_fraction": 0.20,
    "rul_cap": 125,
    "rolling_windows": [5, 20],
    "bootstrap_repetitions": 500,
    "bootstrap_confidence": 0.95,
    "shap_max_rows": 500,
    "failure_case_count": 5,
    "runtime_limit_minutes": 45,
    "download_timeout_seconds": 180,
    "clean_output_dir": True,
    "xgb_parameters": {
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.05,
        "reg_lambda": 1.0,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "n_jobs": -1,
    },
}

PACKAGE_SPECS = {
    "numpy": ("numpy", "numpy>=1.24,<3"),
    "pandas": ("pandas", "pandas>=2.0,<4"),
    "sklearn": ("scikit-learn", "scikit-learn>=1.3,<2"),
    "xgboost": ("xgboost", "xgboost>=2.0,<4"),
    "shap": ("shap", "shap>=0.45,<1"),
    "matplotlib": ("matplotlib", "matplotlib>=3.7,<4"),
    "seaborn": ("seaborn", "seaborn>=0.12,<1"),
    "reportlab": ("reportlab", "reportlab>=4,<5"),
}

os.environ.setdefault("MPLCONFIGDIR", str(CONFIG["base_dir"] / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

missing_specs = []
for import_name, (_, install_spec) in PACKAGE_SPECS.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing_specs.append(install_spec)

if missing_specs:
    print("Installing missing packages:", ", ".join(missing_specs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_specs])

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import sklearn
import xgboost
from IPython.display import display
from reportlab.pdfbase.pdfmetrics import stringWidth
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

data_dir = CONFIG["data_dir"]
output_dir = CONFIG["output_dir"]
data_dir.mkdir(parents=True, exist_ok=True)

protected_paths = {Path("/").resolve(), Path.home().resolve(), CONFIG["base_dir"].resolve()}
if CONFIG["clean_output_dir"] and output_dir.exists():
    if output_dir.resolve() in protected_paths:
        raise ValueError("Refusing to clean a broad output path. Choose a dedicated output directory.")
    for child in output_dir.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
output_dir.mkdir(parents=True, exist_ok=True)

version_rows = []
for import_name, (package_name, _) in PACKAGE_SPECS.items():
    module = importlib.import_module(import_name)
    version_rows.append({"package": package_name, "version": getattr(module, "__version__", "unknown")})
version_rows.append({"package": "python", "version": sys.version.split()[0]})
display(pd.DataFrame(version_rows))
print(f"Data directory: {data_dir.resolve()}")
print(f"Output directory: {output_dir.resolve()}")

        package  version
0         numpy    2.4.6
1        pandas    3.0.5
2  scikit-learn    1.9.0
3       xgboost    3.3.0
4          shap   0.52.0
5    matplotlib   3.11.1
6       seaborn   0.13.2
7     reportlab    5.0.0
8        python  3.12.13
Data directory: /workspace/scratch/45a1484d94ba/predictive_maintenance_data
Output directory: /workspace/scratch/45a1484d94ba/predictive_maintenance_outputs


## Data acquisition, formulas, and preflight tests

The parser expects 26 columns: engine ID, cycle, three operating settings, and 21 sensors. RUL for a training row is `engine final cycle - current cycle`, optionally capped at the configured value. Rolling means and standard deviations are grouped by engine and use only the current and preceding rows.

For an error \(d_i = \hat{y}_i-y_i\), the asymmetric NASA penalty is

\[
s_i =
\begin{cases}
\exp(-d_i/13)-1, & d_i < 0 \\
\exp(d_i/10)-1, & d_i \ge 0.
\end{cases}
\]

Late predictions receive the larger penalty. The total NASA score is \(\sum_i s_i\), so lower is better. The next cell tests the parser, RUL construction, rolling isolation, the score formula, and a small end-to-end model path before the full data workflow.

In [ ]:
from io import StringIO

SETTING_COLUMNS = [f"setting_{number}" for number in range(1, 4)]
SENSOR_COLUMNS = [f"sensor_{number}" for number in range(1, 22)]
ALL_COLUMNS = ["engine_id", "cycle", *SETTING_COLUMNS, *SENSOR_COLUMNS]


def parse_cmapss_table(source):
    frame = pd.read_csv(source, sep=r"\s+", header=None)
    if frame.shape[1] != len(ALL_COLUMNS):
        raise ValueError(f"Expected 26 columns, found {frame.shape[1]}.")
    frame.columns = ALL_COLUMNS
    for key in ["engine_id", "cycle"]:
        if not np.allclose(frame[key], np.round(frame[key])):
            raise ValueError(f"{key} must contain integer values.")
        frame[key] = frame[key].astype(int)
    return frame.sort_values(["engine_id", "cycle"]).reset_index(drop=True)


def add_training_rul(frame, cap):
    result = frame[["engine_id", "cycle"]].copy()
    final_cycle = frame.groupby("engine_id")["cycle"].transform("max")
    result["rul_raw"] = final_cycle - frame["cycle"]
    result["rul_target"] = result["rul_raw"].clip(upper=cap) if cap is not None else result["rul_raw"]
    return result


def build_cycle_safe_features(frame, sensor_columns, setting_columns, windows):
    ordered = frame.sort_values(["engine_id", "cycle"]).reset_index(drop=True)
    feature_parts = [ordered[["cycle", *setting_columns, *sensor_columns]].astype(float)]
    for window in windows:
        grouped = ordered.groupby("engine_id", sort=False)[sensor_columns]
        rolling = grouped.rolling(window=window, min_periods=1)
        rolling_mean = rolling.mean().reset_index(level=0, drop=True)
        rolling_std = rolling.std(ddof=0).reset_index(level=0, drop=True)
        rolling_mean.columns = [f"{name}_mean_{window}" for name in sensor_columns]
        rolling_std.columns = [f"{name}_std_{window}" for name in sensor_columns]
        feature_parts.extend([rolling_mean, rolling_std])
    features = pd.concat(feature_parts, axis=1)
    return pd.concat([ordered[["engine_id"]], features], axis=1)


def nasa_score(y_true, y_pred):
    errors = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    penalties = np.where(errors < 0, np.exp(-errors / 13.0) - 1.0, np.exp(errors / 10.0) - 1.0)
    return float(np.sum(penalties))


def compute_metrics(y_true, y_pred):
    values = {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "nasa_score": nasa_score(y_true, y_pred),
    }
    if not np.isfinite(list(values.values())).all():
        raise ValueError("A metric is not finite.")
    return values


def make_models(seed, xgb_parameters, smoke=False):
    tree_parameters = dict(xgb_parameters)
    tree_parameters["random_state"] = seed
    if smoke:
        tree_parameters.update({"n_estimators": 8, "max_depth": 2, "n_jobs": 1})
    linear = Pipeline(
        [
            ("variance", VarianceThreshold()),
            ("scale", StandardScaler()),
            ("model", RidgeCV(alphas=np.logspace(-3, 3, 13))),
        ]
    )
    tree = Pipeline(
        [
            ("variance", VarianceThreshold()),
            ("model", XGBRegressor(**tree_parameters)),
        ]
    )
    return {"Ridge": linear, "XGBoost": tree}


unit_tests = {}

parser_values = " ".join(str(value) for value in range(1, 27))
parsed_row = parse_cmapss_table(StringIO(parser_values + "\n"))
unit_tests["parser_schema"] = parsed_row.shape == (1, 26) and parsed_row.columns.tolist() == ALL_COLUMNS

toy_rul_frame = pd.DataFrame({"engine_id": [1, 1, 1, 2, 2], "cycle": [1, 2, 3, 1, 2]})
toy_rul = add_training_rul(toy_rul_frame, cap=1)
unit_tests["rul_construction"] = (
    toy_rul["rul_raw"].tolist() == [2, 1, 0, 1, 0]
    and toy_rul["rul_target"].tolist() == [1, 1, 0, 1, 0]
)

toy_features_frame = pd.DataFrame(
    {
        "engine_id": [1, 1, 2, 2],
        "cycle": [1, 2, 1, 2],
        "sensor_1": [1.0, 3.0, 100.0, 200.0],
        "setting_1": [0.0, 0.0, 0.0, 0.0],
    }
)
toy_features = build_cycle_safe_features(toy_features_frame, ["sensor_1"], ["setting_1"], [2])
expected_means = [1.0, 2.0, 100.0, 150.0]
unit_tests["rolling_engine_isolation"] = np.allclose(toy_features["sensor_1_mean_2"], expected_means)

changed_future = toy_features_frame.copy()
changed_future.loc[(changed_future["engine_id"] == 1) & (changed_future["cycle"] == 2), "sensor_1"] = 9999.0
changed_features = build_cycle_safe_features(changed_future, ["sensor_1"], ["setting_1"], [2])
unit_tests["rolling_ignores_future"] = np.isclose(
    toy_features.loc[0, "sensor_1_mean_2"], changed_features.loc[0, "sensor_1_mean_2"]
)

worked_true = np.array([100.0, 100.0, 100.0])
worked_pred = np.array([90.0, 100.0, 110.0])
worked_expected = (np.exp(10.0 / 13.0) - 1.0) + (np.exp(10.0 / 10.0) - 1.0)
unit_tests["nasa_score_formula"] = math.isclose(
    nasa_score(worked_true, worked_pred), worked_expected, rel_tol=1e-12
)

smoke_rows = []
for engine_id in range(1, 5):
    for cycle in range(1, 7):
        smoke_rows.append(
            {
                "engine_id": engine_id,
                "cycle": cycle,
                "sensor_1": engine_id + cycle / 10,
                "sensor_2": 2 * engine_id - cycle / 20,
                "setting_1": 0.0,
            }
        )
smoke_frame = pd.DataFrame(smoke_rows)
smoke_features = build_cycle_safe_features(
    smoke_frame, ["sensor_1", "sensor_2"], ["setting_1"], [2]
)
smoke_targets = add_training_rul(smoke_frame, cap=5)
smoke_data = smoke_features.merge(smoke_targets, on=["engine_id", "cycle"], validate="one_to_one")
smoke_train = smoke_data["engine_id"].isin([1, 2, 3])
smoke_columns = [
    column
    for column in smoke_features.columns
    if column not in {"engine_id"}
]
smoke_predictions = {}
for model_name, model in make_models(CONFIG["seed"], CONFIG["xgb_parameters"], smoke=True).items():
    model.fit(smoke_data.loc[smoke_train, smoke_columns], smoke_data.loc[smoke_train, "rul_target"])
    prediction = np.clip(model.predict(smoke_data.loc[~smoke_train, smoke_columns]), 0, None)
    smoke_predictions[model_name] = prediction
unit_tests["end_to_end_smoke"] = all(np.isfinite(values).all() for values in smoke_predictions.values())

if not all(unit_tests.values()):
    failed_tests = [name for name, result in unit_tests.items() if not result]
    raise AssertionError(f"Preflight test failure: {failed_tests}")

display(pd.DataFrame({"test": unit_tests.keys(), "status": ["PASS"] * len(unit_tests)}))
print("Preflight tests completed successfully.")

                       test status
0             parser_schema   PASS
1          rul_construction   PASS
2  rolling_engine_isolation   PASS
3    rolling_ignores_future   PASS
4        nasa_score_formula   PASS
5          end_to_end_smoke   PASS
Preflight tests completed successfully.


In [ ]:
def find_expected_files(folder, expected_names):
    folder = Path(folder)
    if not folder.exists():
        return None
    resolved = {}
    for name in expected_names:
        matches = list(folder.rglob(name))
        if len(matches) != 1:
            return None
        resolved[name] = matches[0]
    return resolved


def extract_fd001(zip_path, destination, expected_names):
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"Not a readable ZIP archive: {zip_path}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        matches = {}
        for expected in expected_names:
            members = [member for member in archive.namelist() if Path(member).name == expected]
            if len(members) != 1:
                raise ValueError(f"Archive must contain exactly one {expected}; found {len(members)}.")
            matches[expected] = members[0]
        for expected, member in matches.items():
            target = destination / expected
            with archive.open(member) as source, target.open("wb") as sink:
                shutil.copyfileobj(source, sink)
    return {name: destination / name for name in expected_names}


def download_official_zip(url, target, timeout):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response, temporary.open("wb") as sink:
            shutil.copyfileobj(response, sink)
        if temporary.stat().st_size < 1_000_000:
            raise ValueError("Downloaded file is unexpectedly small.")
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()
    return target


def acquire_fd001(config):
    mode = config["data_mode"]
    allowed_modes = {"auto", "official", "manual_zip", "manual_folder"}
    if mode not in allowed_modes:
        raise ValueError(f"data_mode must be one of {sorted(allowed_modes)}.")

    direct_files = find_expected_files(config["data_dir"], config["expected_files"])
    if direct_files and mode == "auto":
        return direct_files, "existing extracted cache"

    if mode == "auto" and config["cache_zip"].exists():
        return (
            extract_fd001(config["cache_zip"], config["data_dir"], config["expected_files"]),
            "cached official ZIP",
        )

    official_error = None
    if mode in {"auto", "official"}:
        try:
            downloaded = download_official_zip(
                config["nasa_zip_url"], config["cache_zip"], config["download_timeout_seconds"]
            )
            return (
                extract_fd001(downloaded, config["data_dir"], config["expected_files"]),
                "NASA Open Data ZIP",
            )
        except Exception as exc:
            official_error = f"{type(exc).__name__}: {exc}"
            if mode == "official":
                raise RuntimeError(f"Official download failed: {official_error}") from exc

    if mode in {"auto", "manual_folder"}:
        manual_files = find_expected_files(config["manual_folder"], config["expected_files"])
        if manual_files:
            return manual_files, "manual FD001 folder"
        if mode == "manual_folder":
            raise FileNotFoundError(
                f"Expected {config['expected_files']} under {config['manual_folder'].resolve()}."
            )

    if mode in {"auto", "manual_zip"} and config["manual_zip"].exists():
        return (
            extract_fd001(config["manual_zip"], config["data_dir"], config["expected_files"]),
            "manual ZIP fallback",
        )
    if mode == "manual_zip":
        raise FileNotFoundError(f"Manual ZIP not found: {config['manual_zip'].resolve()}")

    raise RuntimeError(
        "FD001 acquisition failed. "
        f"Official attempt: {official_error}. "
        f"Place CMAPSSData.zip at {config['manual_zip'].resolve()} or place "
        f"{config['expected_files']} under {config['manual_folder'].resolve()}, then rerun."
    )


fd001_files, data_source = acquire_fd001(CONFIG)
for expected_name, file_path in fd001_files.items():
    if not file_path.exists() or file_path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty required file: {expected_name}")

train_raw = parse_cmapss_table(fd001_files["train_FD001.txt"])
test_raw = parse_cmapss_table(fd001_files["test_FD001.txt"])
rul_values = pd.read_csv(fd001_files["RUL_FD001.txt"], sep=r"\s+", header=None).iloc[:, 0].astype(float)

train_engine_ids_actual = np.sort(train_raw["engine_id"].unique())
test_engine_ids_actual = np.sort(test_raw["engine_id"].unique())
if len(rul_values) != len(test_engine_ids_actual):
    raise ValueError("The number of terminal RUL labels does not match the number of test engines.")

test_labels = pd.DataFrame(
    {"engine_id": test_engine_ids_actual, "terminal_rul": rul_values.to_numpy()}
)
if test_labels["engine_id"].duplicated().any():
    raise ValueError("Duplicate engine IDs in terminal RUL alignment.")

numeric_train = train_raw[ALL_COLUMNS].to_numpy(dtype=float)
numeric_test = test_raw[ALL_COLUMNS].to_numpy(dtype=float)
audit_rows = [
    {"measure": "train rows", "value": len(train_raw)},
    {"measure": "test rows", "value": len(test_raw)},
    {"measure": "train engines", "value": len(train_engine_ids_actual)},
    {"measure": "test engines", "value": len(test_engine_ids_actual)},
    {"measure": "terminal RUL labels", "value": len(test_labels)},
    {"measure": "train duplicate engine-cycle keys", "value": int(train_raw.duplicated(["engine_id", "cycle"]).sum())},
    {"measure": "test duplicate engine-cycle keys", "value": int(test_raw.duplicated(["engine_id", "cycle"]).sum())},
    {"measure": "train missing values", "value": int(train_raw.isna().sum().sum())},
    {"measure": "test missing values", "value": int(test_raw.isna().sum().sum())},
    {"measure": "train finite values", "value": bool(np.isfinite(numeric_train).all())},
    {"measure": "test finite values", "value": bool(np.isfinite(numeric_test).all())},
    {"measure": "minimum terminal RUL", "value": float(test_labels["terminal_rul"].min())},
    {"measure": "maximum terminal RUL", "value": float(test_labels["terminal_rul"].max())},
]
audit_df = pd.DataFrame(audit_rows)

archive_hash = None
if CONFIG["cache_zip"].exists():
    archive_hash = hashlib.sha256(CONFIG["cache_zip"].read_bytes()).hexdigest()
file_hashes = {
    name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in fd001_files.items()
}
data_manifest = {
    "source_mode": data_source,
    "resource_page": CONFIG["nasa_resource_page"],
    "download_url": CONFIG["nasa_zip_url"],
    "cache_zip_sha256": archive_hash,
    "files": {
        name: {
            "path": str(path.resolve()),
            "size_bytes": path.stat().st_size,
            "sha256": file_hashes[name],
        }
        for name, path in fd001_files.items()
    },
    "actual_counts": {
        "train_rows": len(train_raw),
        "test_rows": len(test_raw),
        "train_engines": len(train_engine_ids_actual),
        "test_engines": len(test_engine_ids_actual),
        "terminal_labels": len(test_labels),
    },
}
(output_dir / "data_manifest.json").write_text(json.dumps(data_manifest, indent=2), encoding="utf-8")

print(f"Data source used: {data_source}")
display(audit_df)

Data source used: existing extracted cache
                              measure  value
0                          train rows  20631
1                           test rows  13096
2                       train engines    100
3                        test engines    100
4                 terminal RUL labels    100
5   train duplicate engine-cycle keys      0
6    test duplicate engine-cycle keys      0
7                train missing values      0
8                 test missing values      0
9                 train finite values   True
10                 test finite values   True
11               minimum terminal RUL    7.0
12               maximum terminal RUL  145.0


## Leakage-safe targets, features, and engine split

Training RUL is computed independently inside each engine. The model matrix contains the current cycle, current operating settings and sensors, plus rolling sensor means and standard deviations for the configured windows. It contains no final-cycle, future-cycle, engine ID, or target-derived feature.

Validation engines are held out before either pipeline is fitted. `VarianceThreshold`, scaling for Ridge, and model parameters are therefore learned only from training engines.

In [ ]:
analysis_start = time.perf_counter()

train_targets = add_training_rul(train_raw, CONFIG["rul_cap"])
train_features = build_cycle_safe_features(
    train_raw, SENSOR_COLUMNS, SETTING_COLUMNS, CONFIG["rolling_windows"]
)
test_features = build_cycle_safe_features(
    test_raw, SENSOR_COLUMNS, SETTING_COLUMNS, CONFIG["rolling_windows"]
)

model_data = train_features.merge(
    train_targets, on=["engine_id", "cycle"], how="inner", validate="one_to_one"
)
if len(model_data) != len(train_raw):
    raise ValueError("Training features and RUL targets did not align one-to-one.")

test_with_labels = test_raw.merge(test_labels, on="engine_id", how="left", validate="many_to_one")
test_final_cycle = test_with_labels.groupby("engine_id")["cycle"].transform("max")
test_with_labels["rul_raw"] = test_with_labels["terminal_rul"] + test_final_cycle - test_with_labels["cycle"]
test_feature_data = test_features.merge(
    test_with_labels[["engine_id", "cycle", "rul_raw", "terminal_rul"]],
    on=["engine_id", "cycle"],
    how="inner",
    validate="one_to_one",
)

feature_columns = [
    column
    for column in train_features.columns
    if column not in {"engine_id"}
]
forbidden_tokens = ("rul", "target", "final_cycle", "max_cycle", "lead", "future", "engine_id")
forbidden_features = [
    column
    for column in feature_columns
    if any(token in column.lower() for token in forbidden_tokens)
]
if forbidden_features:
    raise AssertionError(f"Future- or target-derived features detected: {forbidden_features}")

fit_engine_ids, validation_engine_ids = train_test_split(
    train_engine_ids_actual,
    test_size=CONFIG["validation_fraction"],
    random_state=CONFIG["seed"],
    shuffle=True,
)
fit_engine_ids = np.sort(fit_engine_ids)
validation_engine_ids = np.sort(validation_engine_ids)
if set(fit_engine_ids) & set(validation_engine_ids):
    raise AssertionError("Training and validation engine IDs overlap.")

fit_mask = model_data["engine_id"].isin(fit_engine_ids)
validation_mask = model_data["engine_id"].isin(validation_engine_ids)
if not fit_mask.any() or not validation_mask.any():
    raise ValueError("Engine-held-out split produced an empty partition.")

X_fit = model_data.loc[fit_mask, feature_columns]
y_fit = model_data.loc[fit_mask, "rul_target"]
X_validation = model_data.loc[validation_mask, feature_columns]
y_validation = model_data.loc[validation_mask, "rul_target"]
X_full = model_data[feature_columns]
y_full = model_data["rul_target"]
X_test_all = test_feature_data[feature_columns]

split_manifest = pd.DataFrame(
    {
        "engine_id": train_engine_ids_actual,
        "split": [
            "model_fit" if engine_id in set(fit_engine_ids) else "validation"
            for engine_id in train_engine_ids_actual
        ],
    }
)
feature_manifest = {
    "rul_definition": "final training cycle minus current cycle",
    "training_rul_cap": CONFIG["rul_cap"],
    "rolling_windows": CONFIG["rolling_windows"],
    "rolling_direction": "current and past cycles within engine only",
    "feature_count_before_pipeline_selection": len(feature_columns),
    "feature_columns": feature_columns,
    "forbidden_feature_matches": forbidden_features,
    "fit_engine_ids": fit_engine_ids.tolist(),
    "validation_engine_ids": validation_engine_ids.tolist(),
}

print(f"Feature count before pipeline selection: {len(feature_columns)}")
print(f"Fit engines: {len(fit_engine_ids)} | Validation engines: {len(validation_engine_ids)}")
print(f"Training RUL range after cap: {y_full.min():.0f} to {y_full.max():.0f}")

Feature count before pipeline selection: 109
Fit engines: 80 | Validation engines: 20
Training RUL range after cap: 0 to 125


## Model protocol and evaluation

Ridge uses variance filtering, standardization, and cross-validated regularization. XGBoost uses the same variance filtering but does not need scaling. Both models use fixed, predeclared settings and nonnegative prediction clipping. No test label is used for fitting or model selection.

Validation metrics use all capped-RUL rows from held-out engines. The preferred model is selected by mean engine-level RMSE, not test performance. Final models are then refitted on all training engines. Test metrics use the one terminal prediction that aligns with each official RUL label.

In [ ]:
def per_engine_metrics(predictions):
    rows = []
    for engine_id, group in predictions.groupby("engine_id", sort=True):
        metrics = compute_metrics(group["actual_rul"], group["prediction"])
        rows.append({"engine_id": int(engine_id), **metrics})
    return pd.DataFrame(rows)


validation_models = make_models(CONFIG["seed"], CONFIG["xgb_parameters"])
validation_prediction_frames = []
metric_rows = []
engine_metric_frames = []
fit_times = {}

for model_name, model in validation_models.items():
    fit_started = time.perf_counter()
    model.fit(X_fit, y_fit)
    fit_times[f"{model_name}_validation_seconds"] = time.perf_counter() - fit_started
    prediction = np.clip(model.predict(X_validation), 0, None)
    prediction_frame = model_data.loc[
        validation_mask, ["engine_id", "cycle", "rul_raw", "rul_target"]
    ].copy()
    prediction_frame["actual_rul"] = prediction_frame["rul_target"]
    prediction_frame["prediction"] = prediction
    prediction_frame["residual"] = prediction_frame["prediction"] - prediction_frame["actual_rul"]
    prediction_frame["model"] = model_name
    prediction_frame["split"] = "validation_capped_rows"
    validation_prediction_frames.append(prediction_frame)

    metrics = compute_metrics(prediction_frame["actual_rul"], prediction_frame["prediction"])
    engine_metrics = per_engine_metrics(prediction_frame)
    engine_metrics["model"] = model_name
    engine_metrics["split"] = "validation_capped_rows"
    engine_metric_frames.append(engine_metrics)
    metric_rows.append(
        {
            "model": model_name,
            "split": "validation_capped_rows",
            "n_rows": len(prediction_frame),
            "n_engines": prediction_frame["engine_id"].nunique(),
            "engine_macro_rmse": float(engine_metrics["rmse"].mean()),
            **metrics,
        }
    )

validation_predictions = pd.concat(validation_prediction_frames, ignore_index=True)
validation_metrics = pd.DataFrame(metric_rows)
selected_model_name = validation_metrics.sort_values(
    ["engine_macro_rmse", "rmse", "model"]
).iloc[0]["model"]

final_models = make_models(CONFIG["seed"], CONFIG["xgb_parameters"])
for model_name, model in final_models.items():
    fit_started = time.perf_counter()
    model.fit(X_full, y_full)
    fit_times[f"{model_name}_full_seconds"] = time.perf_counter() - fit_started

test_trajectory_frames = []
test_terminal_frames = []
test_metric_rows = []
for model_name, model in final_models.items():
    all_cycle_prediction = np.clip(model.predict(X_test_all), 0, None)
    trajectory = test_feature_data[["engine_id", "cycle", "rul_raw", "terminal_rul"]].copy()
    trajectory["actual_rul"] = trajectory["rul_raw"]
    trajectory["prediction"] = all_cycle_prediction
    trajectory["residual"] = trajectory["prediction"] - trajectory["actual_rul"]
    trajectory["model"] = model_name
    test_trajectory_frames.append(trajectory)

    terminal = (
        trajectory.sort_values(["engine_id", "cycle"])
        .groupby("engine_id", as_index=False)
        .tail(1)
        .copy()
    )
    terminal["actual_rul"] = terminal["terminal_rul"]
    terminal["residual"] = terminal["prediction"] - terminal["actual_rul"]
    terminal["absolute_error"] = terminal["residual"].abs()
    terminal["split"] = "test_terminal_raw"
    if len(terminal) != len(test_labels) or terminal["engine_id"].nunique() != len(test_labels):
        raise AssertionError(f"{model_name} test predictions do not align one-to-one with labels.")
    aligned = terminal.sort_values("engine_id")["actual_rul"].to_numpy()
    if not np.allclose(aligned, test_labels.sort_values("engine_id")["terminal_rul"].to_numpy()):
        raise AssertionError(f"{model_name} terminal RUL alignment changed.")
    test_terminal_frames.append(terminal)

    metrics = compute_metrics(terminal["actual_rul"], terminal["prediction"])
    test_metric_rows.append(
        {
            "model": model_name,
            "split": "test_terminal_raw",
            "n_rows": len(terminal),
            "n_engines": terminal["engine_id"].nunique(),
            "engine_macro_rmse": np.nan,
            **metrics,
        }
    )

test_trajectories = pd.concat(test_trajectory_frames, ignore_index=True)
test_terminal_predictions = pd.concat(test_terminal_frames, ignore_index=True)
metrics_df = pd.concat(
    [validation_metrics, pd.DataFrame(test_metric_rows)], ignore_index=True
)
engine_level_metrics = pd.concat(engine_metric_frames, ignore_index=True)

metric_columns = ["rmse", "mae", "r2", "nasa_score"]
if not np.isfinite(metrics_df[metric_columns].to_numpy()).all():
    raise AssertionError("Non-finite model metrics detected.")

comparison = metrics_df.pivot(index="model", columns="split", values="rmse")
display(metrics_df.round(3))
print(f"Selected by validation engine-macro RMSE: {selected_model_name}")

plt.figure(figsize=(7.2, 4.2))
sns.barplot(data=metrics_df, x="split", y="rmse", hue="model")
plt.ylabel("RMSE (cycles)")
plt.xlabel("")
plt.title("Ridge and XGBoost under the same split protocol")
plt.tight_layout()
plt.savefig(output_dir / "model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

     model                   split  n_rows  ...     mae     r2  nasa_score
0    Ridge  validation_capped_rows    4070  ...  12.825  0.851   17474.553
1  XGBoost  validation_capped_rows    4070  ...  10.118  0.875   19413.638
2    Ridge       test_terminal_raw     100  ...  14.852  0.784     655.821
3  XGBoost       test_terminal_raw     100  ...  12.909  0.826     491.452

[4 rows x 9 columns]
Selected by validation engine-macro RMSE: XGBoost


In [ ]:
def bootstrap_engine_metrics(predictions, repetitions, seed):
    rng = np.random.default_rng(seed)
    engine_ids = np.sort(predictions["engine_id"].unique())
    grouped = {
        engine_id: group[["actual_rul", "prediction"]].to_numpy(dtype=float)
        for engine_id, group in predictions.groupby("engine_id", sort=False)
    }
    rows = []
    for repetition in range(repetitions):
        sampled_ids = rng.choice(engine_ids, size=len(engine_ids), replace=True)
        sampled = np.concatenate([grouped[engine_id] for engine_id in sampled_ids], axis=0)
        metrics = compute_metrics(sampled[:, 0], sampled[:, 1])
        rows.append(
            {
                "repetition": repetition,
                "sampled_engines": len(sampled_ids),
                "unique_engines": len(np.unique(sampled_ids)),
                "sampled_rows": len(sampled),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


bootstrap_frames = []
seed_offset = 0
for model_name in ["Ridge", "XGBoost"]:
    validation_subset = validation_predictions.query("model == @model_name")
    test_subset = test_terminal_predictions.query("model == @model_name")
    for split_name, subset in [
        ("validation_capped_rows", validation_subset),
        ("test_terminal_raw", test_subset),
    ]:
        boot = bootstrap_engine_metrics(
            subset,
            CONFIG["bootstrap_repetitions"],
            CONFIG["seed"] + seed_offset,
        )
        seed_offset += 1
        boot["model"] = model_name
        boot["split"] = split_name
        bootstrap_frames.append(boot)

bootstrap_results = pd.concat(bootstrap_frames, ignore_index=True)
alpha = 1.0 - CONFIG["bootstrap_confidence"]
interval_rows = []
for (model_name, split_name), group in bootstrap_results.groupby(["model", "split"]):
    point_row = metrics_df.query("model == @model_name and split == @split_name").iloc[0]
    for metric_name in metric_columns:
        interval_rows.append(
            {
                "model": model_name,
                "split": split_name,
                "metric": metric_name,
                "estimate": float(point_row[metric_name]),
                "lower": float(group[metric_name].quantile(alpha / 2)),
                "upper": float(group[metric_name].quantile(1 - alpha / 2)),
                "confidence": CONFIG["bootstrap_confidence"],
                "method": "percentile cluster bootstrap over engine IDs",
                "repetitions": CONFIG["bootstrap_repetitions"],
            }
        )
bootstrap_intervals = pd.DataFrame(interval_rows)
if not np.isfinite(bootstrap_intervals[["estimate", "lower", "upper"]].to_numpy()).all():
    raise AssertionError("A bootstrap interval is not finite.")

display(
    bootstrap_intervals.query("split == 'test_terminal_raw'")
    .pivot_table(index=["model", "metric"], values=["estimate", "lower", "upper"])
    .round(3)
)

                    estimate    lower    upper
model   metric                                
Ridge   mae           14.852   12.477   17.230
        nasa_score   655.821  449.749  867.888
        r2             0.784    0.702    0.848
        rmse          19.324   16.439   21.985
XGBoost mae           12.909   10.847   15.139
        nasa_score   491.452  337.711  676.700
        r2             0.826    0.746    0.874
        rmse          17.351   14.856   19.817


## Bounded SHAP analysis and failure cases

SHAP is calculated for the refitted XGBoost pipeline on at most the configured number of training rows. The saved matrix retains feature-level SHAP values and row identifiers. This interpretation is associational: correlated sensors and rolling summaries can divide importance among related features.

Failure analysis always uses XGBoost terminal test errors. The five engines are selected by largest absolute terminal error before any plot is created. Each trajectory plot compares model output with RUL reconstructed from the official terminal label.

In [ ]:
xgb_pipeline = final_models["XGBoost"]
xgb_selector = xgb_pipeline.named_steps["variance"]
xgb_model = xgb_pipeline.named_steps["model"]
selected_feature_names = np.asarray(feature_columns)[xgb_selector.get_support()].tolist()

sample_size = min(CONFIG["shap_max_rows"], len(X_full))
sample_indices = (
    model_data.sample(n=sample_size, random_state=CONFIG["seed"]).index.to_numpy()
)
X_shap_original = X_full.loc[sample_indices]
X_shap_transformed = xgb_selector.transform(X_shap_original)
X_shap_frame = pd.DataFrame(
    X_shap_transformed,
    columns=selected_feature_names,
    index=sample_indices,
)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap_frame, check_additivity=False)
shap_values = np.asarray(shap_values)
if shap_values.shape != X_shap_frame.shape or not np.isfinite(shap_values).all():
    raise AssertionError("SHAP matrix shape or values are invalid.")

shap_rows = model_data.loc[sample_indices, ["engine_id", "cycle"]].reset_index(drop=True)
shap_data = pd.concat(
    [
        shap_rows,
        pd.DataFrame(
            shap_values,
            columns=[f"shap__{name}" for name in selected_feature_names],
        ),
    ],
    axis=1,
)
shap_importance = (
    pd.DataFrame(
        {
            "feature": selected_feature_names,
            "mean_absolute_shap": np.abs(shap_values).mean(axis=0),
        }
    )
    .sort_values("mean_absolute_shap", ascending=False)
    .reset_index(drop=True)
)
shap_metadata = {
    "model": "XGBoost refitted on all training engines",
    "sample_rows": sample_size,
    "maximum_rows": CONFIG["shap_max_rows"],
    "selected_feature_count": len(selected_feature_names),
    "background": "TreeExplainer default tree-path-dependent background",
    "interpretation_limit": "Correlated raw and rolling features can share importance.",
}

plt.figure(figsize=(8.5, 6.0))
shap.summary_plot(
    shap_values,
    X_shap_frame,
    max_display=15,
    show=False,
    plot_type="dot",
)
plt.title("XGBoost SHAP summary")
plt.tight_layout()
plt.savefig(output_dir / "shap_summary_beeswarm.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(8.0, 5.5))
shap.summary_plot(
    shap_values,
    X_shap_frame,
    max_display=15,
    show=False,
    plot_type="bar",
)
plt.title("Mean absolute SHAP values")
plt.tight_layout()
plt.savefig(output_dir / "shap_summary_bar.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

display(shap_importance.head(15))

              feature  mean_absolute_shap
0               cycle            8.181382
1     sensor_4_mean_5            7.590963
2    sensor_11_mean_5            5.772266
3    sensor_15_mean_5            4.393458
4     sensor_2_mean_5            3.098542
5   sensor_14_mean_20            2.266751
6     sensor_6_std_20            2.237452
7     sensor_9_mean_5            2.063000
8     sensor_3_mean_5            1.757928
9    sensor_8_mean_20            1.468281
10   sensor_9_mean_20            1.302607
11   sensor_3_mean_20            1.180773
12   sensor_14_std_20            1.159314
13    sensor_8_mean_5            1.130332
14  sensor_12_mean_20            1.126082


predictive_maintenance_baseline_study.ipynb:cell-7:55: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
predictive_maintenance_baseline_study.ipynb:cell-7:69: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.


In [ ]:
xgb_terminal = (
    test_terminal_predictions.query("model == 'XGBoost'")
    .sort_values(["absolute_error", "engine_id"], ascending=[False, True])
    .reset_index(drop=True)
)
failure_cases = xgb_terminal.head(CONFIG["failure_case_count"]).copy()
selected_failure_ids = failure_cases["engine_id"].astype(int).tolist()
if len(selected_failure_ids) != CONFIG["failure_case_count"]:
    raise AssertionError("The required number of failure cases was not selected.")

failure_plot_paths = []
xgb_trajectories = test_trajectories.query("model == 'XGBoost'").copy()
for engine_id in selected_failure_ids:
    engine_data = xgb_trajectories.query("engine_id == @engine_id").sort_values("cycle")
    terminal_row = failure_cases.query("engine_id == @engine_id").iloc[0]
    figure_path = output_dir / f"failure_engine_{engine_id:03d}.png"
    plt.figure(figsize=(7.4, 4.3))
    plt.plot(engine_data["cycle"], engine_data["actual_rul"], label="Actual RUL", linewidth=2)
    plt.plot(engine_data["cycle"], engine_data["prediction"], label="XGBoost prediction", linewidth=2)
    plt.scatter(
        [terminal_row["cycle"]],
        [terminal_row["prediction"]],
        color="tab:red",
        zorder=3,
        label=f"Terminal error: {terminal_row['residual']:+.1f}",
    )
    plt.xlabel("Observed cycle")
    plt.ylabel("Remaining useful life (cycles)")
    plt.title(f"Failure case: test engine {engine_id}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()
    failure_plot_paths.append(figure_path)

plotted_failure_ids = [
    int(path.stem.replace("failure_engine_", "")) for path in failure_plot_paths
]
if plotted_failure_ids != selected_failure_ids:
    raise AssertionError("Failure plots do not match the selected engines in order.")

plt.figure(figsize=(7.0, 4.5))
sns.scatterplot(
    data=xgb_terminal,
    x="prediction",
    y="residual",
    hue="actual_rul",
    palette="viridis",
    s=55,
)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Terminal predicted RUL")
plt.ylabel("Residual: prediction - actual")
plt.title("XGBoost terminal residuals across test engines")
plt.tight_layout()
plt.savefig(output_dir / "global_residual_plot.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

display(
    failure_cases[
        ["engine_id", "cycle", "actual_rul", "prediction", "residual", "absolute_error"]
    ].round(2)
)

   engine_id  cycle  actual_rul  prediction  residual  absolute_error
0         45  152.0       114.0       61.82    -52.18           52.18
1         89  177.0       136.0       89.94    -46.06           46.06
2         72  131.0        50.0       89.80     39.80           39.80
3         74  137.0       126.0       87.58    -38.42           38.42
4         93  244.0        85.0       51.19    -33.81           33.81


## Artifacts, validation, and evidence

The next cell saves the analytical tables and plots, builds a claims table from measured variables, verifies the output contract, and creates three resume bullets. Text width is measured with Times-Roman at 11 points against a 7.12-inch line.

In [ ]:
validation_predictions.to_csv(output_dir / "validation_predictions.csv", index=False)
test_terminal_predictions.to_csv(output_dir / "test_terminal_predictions.csv", index=False)
test_trajectories.query("model == 'XGBoost'").to_csv(
    output_dir / "test_trajectory_predictions.csv", index=False
)
metrics_df.to_csv(output_dir / "metrics.csv", index=False)
engine_level_metrics.to_csv(output_dir / "engine_level_metrics.csv", index=False)
bootstrap_results.to_csv(output_dir / "bootstrap_results.csv", index=False)
bootstrap_intervals.to_csv(output_dir / "bootstrap_intervals.csv", index=False)
shap_data.to_csv(output_dir / "shap_values.csv", index=False)
shap_importance.to_csv(output_dir / "shap_feature_importance.csv", index=False)
(output_dir / "shap_metadata.json").write_text(
    json.dumps(shap_metadata, indent=2), encoding="utf-8"
)
failure_cases.to_csv(output_dir / "failure_cases.csv", index=False)
audit_df.to_csv(output_dir / "data_quality_audit.csv", index=False)
split_manifest.to_csv(output_dir / "split_manifest.csv", index=False)
(output_dir / "feature_manifest.json").write_text(
    json.dumps(feature_manifest, indent=2), encoding="utf-8"
)
(output_dir / "test_results.json").write_text(
    json.dumps(
        {
            "unit_like_tests": {name: bool(result) for name, result in unit_tests.items()},
            "smoke_models": {
                name: values.tolist() for name, values in smoke_predictions.items()
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)


validation_records = []


def record_check(check_name, condition, measured_value, expected_condition, evidence_artifact):
    validation_records.append(
        {
            "check name": check_name,
            "status": "PASS" if bool(condition) else "FAIL",
            "measured value": str(measured_value),
            "expected condition": expected_condition,
            "evidence artifact": evidence_artifact,
        }
    )


record_check(
    "required FD001 files",
    set(fd001_files) == set(CONFIG["expected_files"]),
    sorted(fd001_files),
    "exactly train_FD001.txt, test_FD001.txt, and RUL_FD001.txt",
    "data_manifest.json",
)
record_check(
    "parser schema",
    train_raw.shape[1] == 26 and test_raw.shape[1] == 26,
    f"train={train_raw.shape[1]}, test={test_raw.shape[1]}",
    "26 named columns in each trajectory file",
    "data_quality_audit.csv",
)
record_check(
    "FD001 engine counts",
    len(train_engine_ids_actual) == 100 and len(test_engine_ids_actual) == 100,
    f"train={len(train_engine_ids_actual)}, test={len(test_engine_ids_actual)}",
    "100 train engines and 100 test engines",
    "data_manifest.json",
)
record_check(
    "required data volume",
    len(train_raw) >= 20_000 and len(test_raw) >= 10_000,
    f"train rows={len(train_raw)}, test rows={len(test_raw)}",
    "FD001-scale trajectories with at least 20,000 train and 10,000 test rows",
    "data_quality_audit.csv",
)
record_check(
    "duplicate engine-cycle keys",
    not train_raw.duplicated(["engine_id", "cycle"]).any()
    and not test_raw.duplicated(["engine_id", "cycle"]).any(),
    f"train={train_raw.duplicated(['engine_id', 'cycle']).sum()}, test={test_raw.duplicated(['engine_id', 'cycle']).sum()}",
    "zero duplicate keys",
    "data_quality_audit.csv",
)
record_check(
    "missing and non-finite values",
    train_raw.isna().sum().sum() == 0
    and test_raw.isna().sum().sum() == 0
    and np.isfinite(numeric_train).all()
    and np.isfinite(numeric_test).all(),
    "no missing values; numeric arrays finite",
    "no missing or non-finite trajectory values",
    "data_quality_audit.csv",
)
record_check(
    "RUL construction test",
    unit_tests["rul_construction"],
    unit_tests["rul_construction"],
    "hand-worked raw and capped RUL values match",
    "test_results.json",
)
record_check(
    "rolling-window isolation tests",
    unit_tests["rolling_engine_isolation"] and unit_tests["rolling_ignores_future"],
    {
        "engine_isolation": unit_tests["rolling_engine_isolation"],
        "future_invariance": unit_tests["rolling_ignores_future"],
    },
    "rolling values do not cross engines or use a later cycle",
    "test_results.json",
)
record_check(
    "NASA-score formula test",
    unit_tests["nasa_score_formula"],
    unit_tests["nasa_score_formula"],
    "hand-worked asymmetric penalty matches",
    "test_results.json",
)
record_check(
    "end-to-end smoke test",
    unit_tests["end_to_end_smoke"],
    unit_tests["end_to_end_smoke"],
    "both tiny model paths return finite predictions",
    "test_results.json",
)
record_check(
    "engine split disjointness",
    set(fit_engine_ids).isdisjoint(validation_engine_ids),
    f"intersection={sorted(set(fit_engine_ids) & set(validation_engine_ids))}",
    "no engine ID appears in both fit and validation",
    "split_manifest.csv",
)
record_check(
    "no future-cycle feature",
    len(forbidden_features) == 0 and unit_tests["rolling_ignores_future"],
    f"forbidden matches={forbidden_features}",
    "no target, final-cycle, lead, future, or engine-ID model feature",
    "feature_manifest.json",
)
record_check(
    "test label alignment",
    all(
        len(group) == len(test_labels)
        and group["engine_id"].nunique() == len(test_labels)
        for _, group in test_terminal_predictions.groupby("model")
    ),
    {
        model: len(group)
        for model, group in test_terminal_predictions.groupby("model")
    },
    "one terminal prediction per RUL label for each model",
    "test_terminal_predictions.csv",
)
record_check(
    "finite model metrics",
    np.isfinite(metrics_df[metric_columns].to_numpy()).all(),
    metrics_df[metric_columns].round(4).to_dict(orient="records"),
    "all RMSE, MAE, R2, and NASA scores finite",
    "metrics.csv",
)
record_check(
    "engine-level bootstrap",
    len(bootstrap_results)
    == 4 * CONFIG["bootstrap_repetitions"]
    and (bootstrap_results["sampled_engines"] > 0).all(),
    f"rows={len(bootstrap_results)}, repetitions per model-split={CONFIG['bootstrap_repetitions']}",
    "whole engine IDs resampled for both models and splits",
    "bootstrap_results.csv",
)
record_check(
    "bounded SHAP output",
    len(shap_data) <= CONFIG["shap_max_rows"]
    and len(shap_data) == sample_size
    and np.isfinite(shap_values).all(),
    f"rows={len(shap_data)}, limit={CONFIG['shap_max_rows']}",
    "finite SHAP matrix at or below configured row limit",
    "shap_values.csv",
)
record_check(
    "failure-case selection",
    len(selected_failure_ids) == CONFIG["failure_case_count"]
    and selected_failure_ids
    == xgb_terminal.head(CONFIG["failure_case_count"])["engine_id"].astype(int).tolist(),
    selected_failure_ids,
    "the five largest absolute XGBoost terminal errors",
    "failure_cases.csv",
)
record_check(
    "failure plots match selection",
    plotted_failure_ids == selected_failure_ids
    and all(path.exists() and path.stat().st_size > 0 for path in failure_plot_paths),
    plotted_failure_ids,
    "five plotted engines equal the five selected failure cases in order",
    "failure_engine_*.png",
)

runtime_minutes = (time.perf_counter() - analysis_start) / 60.0
record_check(
    "CPU analysis runtime",
    runtime_minutes < CONFIG["runtime_limit_minutes"],
    f"{runtime_minutes:.2f} minutes after parsing",
    f"under {CONFIG['runtime_limit_minutes']} minutes",
    "results_summary.json",
)

xgb_test_metrics = metrics_df.query(
    "model == 'XGBoost' and split == 'test_terminal_raw'"
).iloc[0]
xgb_test_intervals = bootstrap_intervals.query(
    "model == 'XGBoost' and split == 'test_terminal_raw'"
)
xgb_rmse_interval = xgb_test_intervals.query("metric == 'rmse'").iloc[0]

claims_table = pd.DataFrame(
    [
        {
            "claim": "FD001 engine scale",
            "source variable/file": "train_engine_ids_actual; data_manifest.json",
            "measured value": len(train_engine_ids_actual),
            "validation status": "PASS" if len(train_engine_ids_actual) == 100 else "FAIL",
            "safe wording": f"Modeled remaining useful life for {len(train_engine_ids_actual)} NASA FD001 training engines.",
        },
        {
            "claim": "cycle-safe feature construction",
            "source variable/file": "forbidden_features; feature_manifest.json; test_results.json",
            "measured value": f"{len(feature_columns)} features; forbidden matches={len(forbidden_features)}",
            "validation status": "PASS" if len(forbidden_features) == 0 and unit_tests["rolling_ignores_future"] else "FAIL",
            "safe wording": "Used current and past cycles only within each engine.",
        },
        {
            "claim": "two-model comparison",
            "source variable/file": "metrics.csv",
            "measured value": sorted(metrics_df["model"].unique().tolist()),
            "validation status": "PASS" if set(metrics_df["model"]) == {"Ridge", "XGBoost"} else "FAIL",
            "safe wording": "Compared regularized linear regression with XGBoost under one engine-held-out protocol.",
        },
        {
            "claim": "XGBoost test performance",
            "source variable/file": "metrics.csv; bootstrap_intervals.csv",
            "measured value": f"RMSE={xgb_test_metrics['rmse']:.3f}; NASA={xgb_test_metrics['nasa_score']:.3f}; RMSE 95% CI=[{xgb_rmse_interval['lower']:.3f}, {xgb_rmse_interval['upper']:.3f}]",
            "validation status": "PASS" if np.isfinite(xgb_test_metrics[metric_columns].to_numpy(dtype=float)).all() else "FAIL",
            "safe wording": "Report the measured terminal RMSE and NASA penalty with an engine-bootstrap confidence interval.",
        },
        {
            "claim": "SHAP and failure analysis",
            "source variable/file": "shap_values.csv; failure_cases.csv",
            "measured value": f"SHAP rows={sample_size}; failure cases={len(failure_cases)}",
            "validation status": "PASS" if len(failure_cases) == 5 and len(shap_data) == sample_size else "FAIL",
            "safe wording": f"Explained XGBoost with bounded SHAP and reviewed {len(failure_cases)} terminal failure cases.",
        },
    ]
)
claims_table.to_csv(output_dir / "claims_table.csv", index=False)

core_complete = all(record["status"] == "PASS" for record in validation_records)
line_limit_points = 7.12 * 72


def choose_fitting_bullet(candidates):
    for candidate in candidates:
        width = stringWidth(candidate, "Times-Roman", 11)
        if width <= line_limit_points:
            return candidate, width
    raise AssertionError("No resume bullet candidate fits the required width.")


def build_resume_bullets(use_measured_results):
    engine_count = len(train_engine_ids_actual)
    if use_measured_results:
        candidate_groups = [
            [
                f"Modeled remaining useful life for {engine_count} NASA FD001 engines with cycle-safe rolling features.",
                f"Built cycle-safe RUL features for {engine_count} NASA FD001 turbofan engines.",
            ],
            [
                f"Compared Ridge and XGBoost; XGBoost reached {xgb_test_metrics['rmse']:.1f}-cycle RMSE and {xgb_test_metrics['nasa_score']:,.0f} NASA score.",
                f"Benchmarked Ridge vs. XGBoost at {xgb_test_metrics['rmse']:.1f}-cycle RMSE and {xgb_test_metrics['nasa_score']:,.0f} NASA score.",
            ],
            [
                f"Used SHAP, engine-bootstrap 95% CIs, and {len(failure_cases)} terminal failure cases for error analysis.",
                f"Explained XGBoost with SHAP, engine-bootstrap 95% CIs, and {len(failure_cases)} failure cases.",
            ],
        ]
    else:
        candidate_groups = [
            [
                f"Built a leakage-safe FD001 RUL workflow across {engine_count} NASA turbofan engines.",
                "Built a leakage-safe NASA FD001 remaining-useful-life workflow.",
            ],
            [
                "Compared regularized linear regression with a compact XGBoost baseline.",
                "Compared Ridge and compact XGBoost baselines under one protocol.",
            ],
            [
                "Prepared bounded SHAP and engine-level error analysis pending full validation.",
                "Prepared SHAP and engine-level error analysis pending validation.",
            ],
        ]
    selected = [choose_fitting_bullet(group) for group in candidate_groups]
    return [item[0] for item in selected], [item[1] for item in selected]


resume_bullets, bullet_widths = build_resume_bullets(core_complete)
record_check(
    "resume bullet count and width",
    len(resume_bullets) == 3 and all(width <= line_limit_points for width in bullet_widths),
    {
        "count": len(resume_bullets),
        "widths_points": [round(width, 2) for width in bullet_widths],
        "limit_points": round(line_limit_points, 2),
    },
    "exactly three bullets, each no wider than 7.12 inches at 11-point Times-Roman",
    "resume_bullets.txt",
)

required_existing_artifacts = [
    output_dir / "data_manifest.json",
    output_dir / "data_quality_audit.csv",
    output_dir / "test_results.json",
    output_dir / "feature_manifest.json",
    output_dir / "split_manifest.csv",
    output_dir / "validation_predictions.csv",
    output_dir / "test_terminal_predictions.csv",
    output_dir / "test_trajectory_predictions.csv",
    output_dir / "metrics.csv",
    output_dir / "engine_level_metrics.csv",
    output_dir / "bootstrap_results.csv",
    output_dir / "bootstrap_intervals.csv",
    output_dir / "shap_values.csv",
    output_dir / "shap_feature_importance.csv",
    output_dir / "shap_metadata.json",
    output_dir / "shap_summary_beeswarm.png",
    output_dir / "shap_summary_bar.png",
    output_dir / "failure_cases.csv",
    output_dir / "global_residual_plot.png",
    output_dir / "model_comparison.png",
    output_dir / "claims_table.csv",
    *failure_plot_paths,
]
record_check(
    "required analytical artifacts",
    all(path.exists() and path.stat().st_size > 0 for path in required_existing_artifacts),
    f"{sum(path.exists() and path.stat().st_size > 0 for path in required_existing_artifacts)}/{len(required_existing_artifacts)} present and nonempty",
    "all prediction, metric, bootstrap, SHAP, failure, plot, audit, and claims artifacts present",
    "predictive_maintenance_outputs",
)

final_status = "COMPLETE" if all(record["status"] == "PASS" for record in validation_records) else "INCOMPLETE"
if final_status == "INCOMPLETE" and core_complete:
    resume_bullets, bullet_widths = build_resume_bullets(False)

resume_lines = [CONFIG["project_title"]]
if final_status != "COMPLETE":
    resume_lines.append("DO NOT USE ON RESUME YET")
resume_lines.extend([f"• {bullet}" for bullet in resume_bullets])
(output_dir / "resume_bullets.txt").write_text("\n".join(resume_lines) + "\n", encoding="utf-8")

validation_df = pd.DataFrame(validation_records)
validation_df.to_csv(output_dir / "validation_table.csv", index=False)

corrective_actions = [
    f"{row['check name']}: satisfy {row['expected condition']} (measured: {row['measured value']})."
    for row in validation_records
    if row["status"] == "FAIL"
]

metrics_nested = {}
for _, row in metrics_df.iterrows():
    metrics_nested.setdefault(row["model"], {})[row["split"]] = {
        metric_name: float(row[metric_name]) for metric_name in metric_columns
    }
    metrics_nested[row["model"]][row["split"]].update(
        {
            "n_rows": int(row["n_rows"]),
            "n_engines": int(row["n_engines"]),
            "engine_macro_rmse": None
            if pd.isna(row["engine_macro_rmse"])
            else float(row["engine_macro_rmse"]),
        }
    )

bootstrap_summary = {}
for _, row in bootstrap_intervals.iterrows():
    key = f"{row['model']}|{row['split']}|{row['metric']}"
    bootstrap_summary[key] = {
        "estimate": float(row["estimate"]),
        "lower": float(row["lower"]),
        "upper": float(row["upper"]),
        "confidence": float(row["confidence"]),
        "method": row["method"],
        "repetitions": int(row["repetitions"]),
    }

expected_artifact_names = sorted(
    {
        path.name
        for path in [
            *required_existing_artifacts,
            output_dir / "resume_bullets.txt",
            output_dir / "validation_table.csv",
            output_dir / "results_summary.json",
        ]
    }
)
results_summary = {
    "project_title": CONFIG["project_title"],
    "status": final_status,
    "methodology": (
        "FD001-only engine-held-out comparison of Ridge and XGBoost using capped training RUL, "
        "current/past rolling sensor features, terminal test evaluation, and engine-cluster bootstrap intervals."
    ),
    "data_source": data_source,
    "engine_counts": {
        "training": len(train_engine_ids_actual),
        "test": len(test_engine_ids_actual),
        "model_fit": len(fit_engine_ids),
        "validation": len(validation_engine_ids),
    },
    "row_counts": {"training": len(train_raw), "test": len(test_raw)},
    "training_rul_cap": CONFIG["rul_cap"],
    "feature_count": len(feature_columns),
    "selected_model_by_validation_engine_macro_rmse": selected_model_name,
    "model_metrics": metrics_nested,
    "bootstrap_intervals": bootstrap_summary,
    "shap_sample_rows": sample_size,
    "shap_selected_feature_count": len(selected_feature_names),
    "failure_case_model": "XGBoost",
    "failure_case_count": len(failure_cases),
    "failure_case_engine_ids": selected_failure_ids,
    "resume_bullets": resume_bullets,
    "resume_bullet_widths_points": [float(width) for width in bullet_widths],
    "resume_line_limit_points": line_limit_points,
    "validation_pass_count": int((validation_df["status"] == "PASS").sum()),
    "validation_check_count": len(validation_df),
    "runtime_minutes_after_parsing": runtime_minutes,
    "fit_times_seconds": fit_times,
    "limitations": [
        "FD001 is simulated and contains one condition and one fault mode.",
        "The capped target limits early-life RUL extrapolation.",
        "Terminal test labels support direct evaluation at one cycle per engine.",
        "Bootstrap confidence intervals are not per-engine predictive intervals.",
    ],
    "corrective_actions": corrective_actions,
    "claims_table_path": str((output_dir / "claims_table.csv").resolve()),
    "validation_table_path": str((output_dir / "validation_table.csv").resolve()),
    "artifacts": expected_artifact_names,
    "completed_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
(output_dir / "results_summary.json").write_text(
    json.dumps(results_summary, indent=2), encoding="utf-8"
)

final_required = [
    output_dir / "results_summary.json",
    output_dir / "resume_bullets.txt",
    output_dir / "validation_table.csv",
]
if not all(path.exists() and path.stat().st_size > 0 for path in final_required):
    raise AssertionError("A final contract file was not saved.")

print(f"Saved {len(expected_artifact_names)} expected artifacts.")

Saved 29 expected artifacts.


## Evidence & Resume

This final cell reads the saved summary, validation table, claims table, and bullet file. It does not hard-code project metrics.

In [ ]:
saved_summary = json.loads((output_dir / "results_summary.json").read_text(encoding="utf-8"))
saved_validation = pd.read_csv(output_dir / "validation_table.csv")
saved_claims = pd.read_csv(output_dir / "claims_table.csv")
saved_bullet_lines = (output_dir / "resume_bullets.txt").read_text(encoding="utf-8").splitlines()

required_final_keys = [
    "project_title",
    "status",
    "methodology",
    "data_source",
    "engine_counts",
    "row_counts",
    "training_rul_cap",
    "feature_count",
    "selected_model_by_validation_engine_macro_rmse",
    "model_metrics",
    "bootstrap_intervals",
    "shap_sample_rows",
    "failure_case_count",
    "failure_case_engine_ids",
    "resume_bullets",
    "resume_bullet_widths_points",
    "validation_pass_count",
    "validation_check_count",
    "runtime_minutes_after_parsing",
    "limitations",
    "corrective_actions",
    "artifacts",
]
missing_final_keys = [key for key in required_final_keys if key not in saved_summary]
if missing_final_keys:
    raise KeyError(f"Missing critical result keys: {missing_final_keys}")

xgb_saved = saved_summary["model_metrics"]["XGBoost"]["test_terminal_raw"]
print(saved_summary["project_title"])
print(saved_summary["methodology"])
print(
    f"XGBoost terminal test results: RMSE={xgb_saved['rmse']:.3f} cycles, "
    f"MAE={xgb_saved['mae']:.3f}, R2={xgb_saved['r2']:.3f}, "
    f"NASA score={xgb_saved['nasa_score']:.3f}."
)
print(
    f"Validation-selected model: {saved_summary['selected_model_by_validation_engine_macro_rmse']}; "
    f"SHAP rows: {saved_summary['shap_sample_rows']}; "
    f"failure cases: {saved_summary['failure_case_count']}."
)

print("\nValidation table")
display(saved_validation)

print("\nClaims table")
display(saved_claims)

print("\nFinal resume bullets and measured widths")
bullet_width_table = pd.DataFrame(
    {
        "bullet": saved_summary["resume_bullets"],
        "width_points": saved_summary["resume_bullet_widths_points"],
        "limit_points": saved_summary["resume_line_limit_points"],
    }
)
for bullet, width in zip(
    saved_summary["resume_bullets"], saved_summary["resume_bullet_widths_points"]
):
    print(f"• {bullet} [{width:.2f} pt]")
display(bullet_width_table)

artifact_rows = []
for artifact_path in sorted(output_dir.iterdir()):
    if artifact_path.is_file():
        artifact_rows.append(
            {
                "path": str(artifact_path.resolve()),
                "size_bytes": artifact_path.stat().st_size,
            }
        )
artifact_inventory = pd.DataFrame(artifact_rows)
print("\nGenerated artifacts")
display(artifact_inventory)

print(f"\nFINAL STATUS: {saved_summary['status']}")
if saved_summary["corrective_actions"]:
    print("Remaining manual actions:")
    for action in saved_summary["corrective_actions"]:
        print(f"- {action}")
else:
    print("Remaining manual actions: none.")

if saved_summary["status"] == "COMPLETE":
    if not (saved_validation["status"] == "PASS").all():
        raise AssertionError("Saved status is COMPLETE but at least one validation check failed.")
else:
    print("Resume file warning:", next((line for line in saved_bullet_lines if line.startswith("DO NOT USE")), "present"))

Predictive Maintenance Baseline Study
FD001-only engine-held-out comparison of Ridge and XGBoost using capped training RUL, current/past rolling sensor features, terminal test evaluation, and engine-cluster bootstrap intervals.
XGBoost terminal test results: RMSE=17.351 cycles, MAE=12.909, R2=0.826, NASA score=491.452.
Validation-selected model: XGBoost; SHAP rows: 500; failure cases: 5.

Validation table
                        check name  ...               evidence artifact
0             required FD001 files  ...              data_manifest.json
1                    parser schema  ...          data_quality_audit.csv
2              FD001 engine counts  ...              data_manifest.json
3             required data volume  ...          data_quality_audit.csv
4      duplicate engine-cycle keys  ...          data_quality_audit.csv
5    missing and non-finite values  ...          data_quality_audit.csv
6            RUL construction test  ...               test_results.json
7   rolling-win